# PROCESSING ACTUATOR DATA

Manual adjustments to actuator data, which was not tracked directly by the PCS, is inserted here.  

**Requirements** 
- Information on Changes in actuator data
- Final actuator data from experiments

**Load necessary packages**

In [ ]:
import os
import pandas as pd
from pathlib  import Path

**Define Relevant Variables**

In [ ]:
actuator_data = Path.cwd().parent / "data" / "02_Timeseries_Acutators" / "final" / "Operation"

# List of H002 entries to be included in the data
changes = { "208": {    "a": {"feature": "H002", "Start": "12:00:30", "End": "12:00:59", "Value": 100}},
            "218": {    "a": {"feature": "H002", "Start": "13:26:00", "End": "13:31:07", "Value": 100},
                        "b": {"feature": "H002", "Start": "14:02:20", "End": "14:06:48", "Value": 50}}}

The changes in the actuator settings are simply saved instead of the originally tracked values.

In [ ]:
for key in changes:
    print(f"Processing changes for {key}")
    for sub_key in changes[key]:
        feature    = changes[key][sub_key]["feature"]
        start_time = changes[key][sub_key]["Start"]
        end_time   = changes[key][sub_key]["End"]
        value      = changes[key][sub_key]["Value"]

        # Create the file name
        file_name = f"{key}.csv"
        file_path = actuator_data / file_name

        # Check if the file exists and load it
        if os.path.exists(file_path):
            df = pd.read_csv(file_path)

            # Check if df['Time'] is in datetime format
            if not pd.api.types.is_datetime64_any_dtype(df['Time']):
                df['Time'] = pd.to_datetime(df['Time'], errors='coerce')
                df['Time'] = df['Time'].dt.time

            # Convert 'Time' column to datetime
            df['Time'] = pd.to_datetime(df['Time'], format='%H:%M:%S')

            # Create a mask for the time range
            mask =  (df['Time'] >= pd.to_datetime(start_time, format='%H:%M:%S')) & \
                    (df['Time'] <= pd.to_datetime(end_time, format='%H:%M:%S'))

            # Update the 'Value' column where the mask is True
            df.loc[mask, feature] = value
            print(f"Updated {feature} from {start_time} to {end_time} with value {value} in {file_name}")

            # Save the updated DataFrame back to CSV
            df.to_csv(file_path, index=False)